# 09 · Combine schools, medical and emergency into one site list

**Replaces** the combining half of the old `build_trimmed_csvs.py`.

**Reads** the three outputs of nb 07/08. **Writes** `data/processed/services_sites.csv`,
which nb 11 and the Streamlit app (`app/app.py`) read.

**Schema change.** The old file packed different things into shared columns. `region_or_class`
held the education region for schools but the facility class for everything else, and
`sector_or_agency` was empty for all medical rows. Each column now means the same thing for every row:

| column | school | medical | emergency |
|---|---|---|---|
| `type` | stages offered (or `Specialist / other`) | NHSD service type | facility class |
| `facility_class` | `School` | GP / Pharmacy / Hospital | Policing / SES / Fire / Ambulance |
| `operator` | sector (Government, Catholic, ...) | *(not in source)* | agency |
| `region` | DoE region | *(n/a)* | *(n/a)* |
| `suburb`, `postcode` | locality / – | from source | from source |
| `geocode_precision` | `site` / `sibling_school` (ACARA), else suburb / community / region / none | `site` | `site` |

The app only uses `source`, `name`, `latitude`, `longitude` and `type`, and all of those are unchanged.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Walk up from wherever Jupyter was started until we find the repo root
# (the folder that holds data/raw), so the notebook runs from any cwd.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
INPUTS = ROOT / "data" / "external" / "services" / "trimmed"   # read-only inputs
PROCESSED = ROOT / "data" / "processed"                         # the only folder we write to

# NT bounding box used as a sanity check on every coordinate we write.
NT_LAT, NT_LON = (-26.1, -10.9), (128.9, 138.1)


def in_nt(df: pd.DataFrame) -> pd.Series:
    return df["latitude"].between(*NT_LAT) & df["longitude"].between(*NT_LON)


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (vectorised; any argument may be an array)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0088 * np.arcsin(np.sqrt(a))

import re


## 1. Load the three service lists

In [2]:
schools = pd.read_csv(PROCESSED / "school_services.csv")
med = pd.read_csv(PROCESSED / "medical_services.csv", dtype={"medical_postcode": str})
emg = pd.read_csv(PROCESSED / "emergency_services.csv", dtype={"emergency_postcode": str})
print(len(schools), "schools |", len(med), "medical |", len(emg), "emergency")


273 schools | 181 medical | 151 emergency


## 2. Map each source onto the common schema

Schools that offer none of the four standard stages (language centres, the hospital school,
flexible-learning centres) get `type = "Specialist / other"` instead of a blank.

In [3]:
def slugify(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", str(text).lower()).strip("-")


STAGES = [("Pre School", "school_is_preschool"), ("Primary", "school_is_primary"),
          ("Middle", "school_is_middle"), ("Senior", "school_is_senior")]

school_sites = pd.DataFrame({
    "service_site_id": [f"school-{slugify(n)}" for n in schools["name"]],   # names are unique
    "source": "school",
    "name": schools["name"],
    "latitude": schools["latitude"],
    "longitude": schools["longitude"],
    "geocode_method": schools["school_geocode_method"],
    "geocode_precision": schools["school_geocode_precision"],
    "type": schools.apply(lambda r: "; ".join(s for s, col in STAGES if r[col]) or "Specialist / other", axis=1),
    "facility_class": "School",
    "operator": schools["school_sector"],
    "region": schools["school_region"],
    "suburb": schools["school_locality"],
    "postcode": None,
})

medical_sites = pd.DataFrame({
    "service_site_id": [f"medical-{i}" for i in range(len(med))],
    "source": "medical",
    "name": med["name"],
    "latitude": med["latitude"],
    "longitude": med["longitude"],
    "geocode_method": "source_supplied",
    "geocode_precision": "site",
    "type": med["medical_service_type"],
    "facility_class": med["medical_class"],
    "operator": None,
    "region": None,
    "suburb": med["medical_suburb"],
    "postcode": med["medical_postcode"],
})

emergency_sites = pd.DataFrame({
    "service_site_id": [f"emergency-{i}" for i in range(len(emg))],
    "source": "emergency",
    "name": emg["name"],
    "latitude": emg["latitude"],
    "longitude": emg["longitude"],
    "geocode_method": "source_supplied",
    "geocode_precision": "site",
    "type": emg["emergency_class"],
    "facility_class": emg["emergency_class"],
    "operator": emg["emergency_agency"],
    "region": None,
    "suburb": emg["emergency_suburb"],
    "postcode": emg["emergency_postcode"],
})

combined = pd.concat([school_sites, medical_sites, emergency_sites], ignore_index=True)


## 3. Validate

In [4]:
assert combined["service_site_id"].is_unique
assert combined["type"].notna().all(), "every site should have a type"
has_xy = combined["latitude"].notna()
assert in_nt(combined[has_xy]).all()

print(combined["source"].value_counts().to_string(), "\n")
print(f"rows with coordinates: {int(has_xy.sum())}/{len(combined)}")
print(combined.groupby("source")["geocode_precision"].value_counts().to_string())


source
school       273
medical      181
emergency    151 

rows with coordinates: 605/605
source     geocode_precision
emergency  site                 151
medical    site                 181
school     site                 200
           sibling_school        56
           suburb                15
           community              1
           region                 1


## 4. Write output

In [5]:
out = PROCESSED / "services_sites.csv"
combined.to_csv(out, index=False)
print(f"wrote {out.relative_to(ROOT)} ({len(combined)} rows)")


wrote data\processed\services_sites.csv (605 rows)
